# Prod MLflow Tracking V3 API Display

Pulls the public telemetry export API and displays the result directly in the notebook.

In [1]:
import datetime as dt
import json
import os
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd

try:
    from pyspark.sql import functions as F
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType
except ImportError:
    F = None
    BooleanType = DoubleType = LongType = StringType = StructField = StructType = None


class _LocalWidgets:
    def __init__(self):
        self._defaults = {}

    def text(self, name: str, default_value: str, label: str | None = None) -> None:
        self._defaults[name] = default_value

    def dropdown(self, name: str, default_value: str, choices: list[str], label: str | None = None) -> None:
        self._defaults[name] = default_value

    def get(self, name: str) -> str:
        env_name = f"NOTEBOOK_{name.upper()}"
        return os.environ.get(env_name, self._defaults.get(name, ""))


class _LocalDbutils:
    widgets = _LocalWidgets()


try:
    dbutils
except NameError:
    dbutils = _LocalDbutils()

try:
    spark
except NameError:
    spark = None


APP_ENV = "prod"
BASE_URL_DEFAULT = "https://apim-external-cub3dqcrgsdnebcb.a01.azurefd.net/benefits-prd/plan-list/"

dbutils.widgets.text("subscription_key", "651d20d2dc0243b19ccc0762e2246e67")
dbutils.widgets.text("base_url", BASE_URL_DEFAULT)
dbutils.widgets.text("start_date", dt.date.today().isoformat())
dbutils.widgets.text("end_date", dt.date.today().isoformat())
dbutils.widgets.text("limit", "10000")
dbutils.widgets.dropdown("include_text", "false", ["false", "true"])
dbutils.widgets.text("app_env_filter", "")

SUBSCRIPTION_KEY = dbutils.widgets.get("subscription_key").strip()
BASE_URL = dbutils.widgets.get("base_url").strip().rstrip("/") + "/"
START_DATE = dt.date.fromisoformat(dbutils.widgets.get("start_date").strip())
END_DATE = dt.date.fromisoformat(dbutils.widgets.get("end_date").strip())
LIMIT = max(1, min(int(dbutils.widgets.get("limit").strip() or "10000"), 10000))
INCLUDE_TEXT = dbutils.widgets.get("include_text").strip().lower() == "true"
APP_ENV_FILTER = dbutils.widgets.get("app_env_filter").strip()

if not SUBSCRIPTION_KEY:
    raise ValueError("Set the subscription_key widget before running the notebook.")
if END_DATE < START_DATE:
    raise ValueError("end_date must be on or after start_date.")


def _fetch_page(event_date: dt.date, cursor: str | None = None) -> dict:
    params = {
        "dataset": "mlflow_tracking_v3",
        "event_date": event_date.isoformat(),
        "limit": str(LIMIT),
        "include_text": "true" if INCLUDE_TEXT else "false",
    }
    if APP_ENV_FILTER:
        params["app_env"] = APP_ENV_FILTER
    if cursor:
        params["cursor"] = cursor

    url = BASE_URL + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY,
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"APIM request failed for {event_date.isoformat()} with {exc.code}: {body[:1000]}"
        ) from exc


def _validate_export_payload(payload: dict, event_date: dt.date) -> None:
    if not isinstance(payload, dict):
        raise RuntimeError(
            f"Unexpected response type for {event_date.isoformat()}: {type(payload).__name__}"
        )
    if "items" not in payload or "has_more" not in payload:
        preview = json.dumps(payload, ensure_ascii=False)[:1000]
        raise RuntimeError(
            "Unexpected export response shape for "
            f"{event_date.isoformat()}: keys={sorted(payload.keys())} preview={preview}"
        )
    if not isinstance(payload.get("items"), list):
        raise RuntimeError(
            f"Unexpected items type for {event_date.isoformat()}: {type(payload.get('items')).__name__}"
        )


def _infer_spark_type(values: list[object]):
    if StringType is None:
        return None
    for value in values:
        if value is None:
            continue
        if isinstance(value, bool):
            return BooleanType()
        if isinstance(value, int) and not isinstance(value, bool):
            return LongType()
        if isinstance(value, float):
            return DoubleType()
        return StringType()
    return StringType()


def _items_to_df(items: list[dict]):
    seen = set()
    columns = []
    for item in items:
        for key in item.keys():
            if key == "cached_input_tokens" or key in seen:
                continue
            seen.add(key)
            columns.append(key)
    if "error_message" in seen:
        columns = [column for column in columns if column != "error_message"] + ["error_message"]
    if "app_env" in seen:
        user_columns = [column for column in ("user_id", "user_name") if column in seen]
        if user_columns:
            columns = [column for column in columns if column not in {"user_id", "user_name"}]
            insert_at = columns.index("app_env") + 1
            columns[insert_at:insert_at] = user_columns
    if spark is None:
        return pd.DataFrame([{column: item.get(column) for column in columns} for item in items], columns=columns)
    schema = StructType(
        [StructField(column, _infer_spark_type([item.get(column) for item in items]), True) for column in columns]
    )
    rows = [
        {column: item.get(column) for column in columns}
        for item in items
    ]
    return spark.createDataFrame(rows, schema=schema)


def _drop_all_null_columns(df):
    if isinstance(df, pd.DataFrame):
        return df.dropna(axis=1, how="all")
    if not df.columns:
        return df
    non_null_flags = (
        df.agg(
            *[
                F.max(F.when(F.col(column).isNotNull(), F.lit(1)).otherwise(F.lit(0))).alias(column)
                for column in df.columns
            ]
        )
        .collect()[0]
        .asDict()
    )
    keep_columns = [column for column in df.columns if non_null_flags.get(column)]
    return df.select(*keep_columns) if keep_columns else df


In [2]:
total_rows = 0
total_pages = 0
day_summaries: list[dict] = []
api_df = None

day = START_DATE
while day <= END_DATE:
    day_rows = 0
    cursor = None
    while True:
        payload = _fetch_page(day, cursor=cursor)
        _validate_export_payload(payload, day)
        items = payload.get("items") or []
        total_pages += 1

        if items:
            batch_df = _items_to_df(items)
            if api_df is None:
                api_df = batch_df
            elif spark is None:
                api_df = pd.concat([api_df, batch_df], ignore_index=True, sort=False)
            else:
                api_df = api_df.unionByName(batch_df, allowMissingColumns=True)
            day_rows += len(items)
            total_rows += len(items)

        if not payload.get("has_more"):
            break
        cursor = payload.get("next_cursor")
        if not cursor:
            raise RuntimeError(f"Missing next_cursor for {day.isoformat()} despite has_more=true.")

    day_summaries.append({"event_date": day.isoformat(), "row_count": day_rows})
    print(f"{day.isoformat()}: {day_rows} rows")
    day += dt.timedelta(days=1)

summary = {
    "app_env": APP_ENV,
    "start_date": START_DATE.isoformat(),
    "end_date": END_DATE.isoformat(),
    "total_rows": total_rows,
    "total_pages": total_pages,
}
print(json.dumps(summary, indent=2))


2026-05-22: 59 rows
{
  "app_env": "prod",
  "start_date": "2026-05-22",
  "end_date": "2026-05-22",
  "total_rows": 59,
  "total_pages": 59
}


In [3]:
if spark is None:
    display(pd.DataFrame(day_summaries))
else:
    display(spark.createDataFrame(day_summaries))

if api_df is None:
    print("No rows returned from the API for the requested window.")
else:
    display(_drop_all_null_columns(api_df))

,event_date,row_count
0,2026-05-22,59


,tracking_id,event_time,question_id,session_id,response_id,facets_product_id,effective_date,response_status,response_time_sec,retry_count,...,user_id,user_name,request_text_preview,request_text_sha256,request_text_char_count,response_text_preview,response_text_sha256,response_text_char_count,total_input_tokens,total_output_tokens
0,99f00a82-bb4e-4ce4-aa8e-e9e8659d5cb9,2026-05-22T13:58:05.112485Z,0ce93527-b6e0-47a0-b054-922129b2c57d,d83913d3-0b21-4370-8648-5374ec64c6a8,resp_0c09ddc9bb596d22006a1060cb38bc8193b35931b...,MG011314,20240101,success,32,0,...,HGAEDE01,HGAEDE01@blueshieldca.com,benefits for physical therapy,a48ba6c8c4c558d66c5a9b9416e09f578a231ba5b57ecf...,29,"**Brief Answer** \nFor **physical therapy**, ...",f0db9243def27cabcf77149fab492e29783b101518cafb...,9180,81059,3130
1,36fb5aa0-ce3c-489e-8230-0de542384565,2026-05-22T13:57:16.779314Z,89014884-72f7-498c-9531-220f277d03f5,d83913d3-0b21-4370-8648-5374ec64c6a8,resp_06773d15beb622d8006a1060add7548196930fe92...,MG011314,20240101,success,13,0,...,HGAEDE01,HGAEDE01@blueshieldca.com,benefits for chiropractic care,646a852fc37051cb7f96231b22f11c6cc235c316fdf794...,30,**Brief Answer** \nChiropractic care (spinal ...,f4e146efd83fc49898c5eb24029360a642d48ba0e14491...,798,81059,606
2,45d6e87c-06ae-4e1b-a2bb-ee4dd8153542,2026-05-22T05:37:02.198244Z,ci-smoke-q-d196137d-eb8c-4e11-9916-97d90a1a1ead,ci-smoke-session-8c047c5f-5865-4f76-a78b-af9a6...,resp_0e468aed6c841fa8006a0feb65ca788190b50e51e...,M0042150,20260101,success,23,0,...,ci-smoke,CI Smoke,What is my copayment or cost for seeing my pri...,dae292e8730498cbed2112d7c3444ec4af6d9a7baa2c86...,72,**Brief Answer** \nFor a primary care physici...,012f118432832bc53843fe87f040d86b6cb564b3a551d7...,3493,112330,1376
3,60fbe176-7408-4f7e-a145-6446fa527d7c,2026-05-22T02:59:38.654228Z,b53d66e9-6738-4578-a269-14e90b37d5cd,922f1e47-d14b-43f9-82d5-3890bee07229,resp_0341d295fd33cca8006a0fc66b2fe481939d48a9c...,ML002186,20260101,success,45,0,...,jherna23,jherna23@blueshieldca.com,Pregnancy NST,b396d03ad8475a997879f797904c388c63305e2ef96997...,13,**Brief Answer** \nA pregnancy NST (non-stres...,16122d6ca91abce00956bb9fb373a580f3e111464a16b1...,5396,109328,3080
4,4b188f48-5d33-4c6b-b87a-2af978bccf1f,2026-05-22T02:58:11.255595Z,96d8b2c7-ff82-4a65-9be1-30e658650984,922f1e47-d14b-43f9-82d5-3890bee07229,resp_03a1bdcf98400625006a0fc636bd588190a973efb...,ML002186,20260101,success,10,0,...,jherna23,jherna23@blueshieldca.com,NOn stress test,6f76baad1490ae136812763447d4d8588761b4b346cf40...,15,**Brief Answer** \nI need one quick clarifica...,67c31ce36c3c3b887b0c93c2711449df494a626eaa28b8...,1074,109329,542
5,dfe14185-ec01-45a9-a244-52b9cf6f5548,2026-05-22T02:51:17.004451Z,fa2f2985-7362-40f8-9243-bae2f12ee589,4dbc7a18-0f19-4d1a-8135-70639b59097e,resp_037c8c5a4a8bf886006a0fc48cff5c819387199d4...,ML002193,20260101,success,22,0,...,lvilla11,lvilla11@blueshieldca.com,deductible medical,83172dcffe448e995456f42af82b92212b4af0f105609b...,18,"**Brief Answer** \nFor this plan, the **in-ne...",aeff38ed12b9fb6b106cb25b12e99383cf656fc2d0b96d...,2375,107042,1014
6,d09fb2a3-c2f2-4f86-b049-29de7c408f95,2026-05-22T02:49:51.575883Z,e3affde9-57e1-48a3-89bb-927aee9f6d3a,80baa288-1806-4732-9a39-8650aee56c66,resp_0a7df06b3a84bce9006a0fc40630f481908a48dfa...,ML002185,20260101,success,71,0,...,bkagui01,bkagui01@blueshieldca.com,tier 4 retail,7018fd6dce71e30e84cabc2198e949e396a3c1ade5c1ec...,13,**Brief Answer** \nFor **Tier 4 Retail (30-da...,5808a162e7e2c933511627c319219ae448781fb2e08329...,3430,109331,4073
7,24d1060c-6175-4d10-ab8e-4d37b3f66811,2026-05-22T02:32:17.056870Z,6101b1fd-9355-405c-881a-2f754dac4f0a,6c629e63-cce0-4106-9e76-bcc4a3fbb5f1,resp_0dac67e0709ef7a1006a0fc027833c81949ec4a7c...,MI011314,20260101,success,7,0,...,cmanec01,cmanec01@blueshieldca.com,medical ophthalmology visit to diagnose/treat ...,992d8f475ed90008e41671e63c10a14bf89bc79e8a345b...,71,"I'm sorry, but I cannot assist with that request.",dd4d79786716bf2be3bbe7f9920b3465f3a48eafb65da5...,49,0,0
8,ee8e